# Análisis del entrenamiento del carrito DQN

El notebook lee un registro existente. Por defecto busca `results/training_log.csv`. Define `CARRITO_TRAINING_LOG` antes de iniciar Jupyter para seleccionar otro CSV. Las rutas relativas se interpretan desde la raíz del proyecto al ejecutar desde la raíz o desde `notebooks/`.

El registro del entrenamiento Base64 conocido documenta una ejecución. Su procedencia es distinta de la de las ejecuciones del lote experimental.


## Preguntas previas

1. ¿Qué esperarías que ocurriera con epsilon?
2. ¿Una recompensa creciente garantiza cero colisiones?
3. ¿Por qué conviene usar una media móvil?

In [ ]:
import os
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
LOG_PATH = ROOT / Path(os.environ.get("CARRITO_TRAINING_LOG", "results/training_log.csv")).expanduser()
OUTPUT_DIR = os.environ.get("CARRITO_NOTEBOOK_OUTPUT")

if not LOG_PATH.is_file():
    raise FileNotFoundError(
        f"No se encontró el registro de entrenamiento en {LOG_PATH}. "
        "Define CARRITO_TRAINING_LOG con la ruta de un training_log.csv existente."
    )

logs = pd.read_csv(LOG_PATH)
required = {"episode", "reward", "steps", "success", "collision", "epsilon", "loss_mean"}
missing = required - set(logs.columns)
if missing:
    raise ValueError(f"Faltan columnas en el registro de entrenamiento: {sorted(missing)}")
if logs.empty:
    raise ValueError("El registro de entrenamiento no contiene episodios.")

figures = {}
display(logs.head())
print("Archivo utilizado:", LOG_PATH)
print("Episodios:", len(logs))


In [ ]:
logs[['episode', 'reward', 'steps', 'success', 'collision', 'epsilon', 'loss_mean']].describe()


## Curvas de aprendizaje

La línea tenue muestra cada episodio; la línea gruesa resume 30 episodios. El suavizado ayuda a ver la tendencia, pero no debe ocultar la variabilidad.

In [ ]:
window = 30
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(logs.episode, logs.reward, alpha=.25, label="episodio")
axes[0].plot(logs.episode, logs.reward.rolling(window, min_periods=1).mean(),
             lw=2, label="media móvil")
axes[0].set_ylabel("Retorno")
axes[0].legend()
axes[0].grid(alpha=.2)
axes[1].plot(logs.episode, logs.success.rolling(window, min_periods=1).mean(),
             lw=2, color="teal")
axes[1].axhline(.90, ls="--", color="coral")
axes[1].set(xlabel="Episodio", ylabel="Tasa de éxito móvil", ylim=(-.02, 1.02))
axes[1].grid(alpha=.2)
fig.tight_layout()
figures["curvas_entrenamiento"] = fig
plt.show()


## Contraste entre el inicio y el final

Se comparan las medias de dos ventanas sin solapamiento, cada una de hasta 50 episodios. La columna `cambio` resta la media inicial de la final.


In [ ]:
n = min(50, len(logs)//2)
comparison = pd.DataFrame({
    'inicio': logs.head(n)[['reward','steps','success','collision']].mean(),
    'final': logs.tail(n)[['reward','steps','success','collision']].mean(),
})
comparison['cambio'] = comparison['final'] - comparison['inicio']
comparison


## Uso de las acciones

In [ ]:
action_columns = [column for column in logs.columns if column.startswith("action_")]
usage = logs[action_columns].sum().sort_values(ascending=False).rename("decisiones")
usage.index.name = "accion"
if not usage.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    usage.plot.bar(ax=ax, color="#2F80ED")
    ax.set(xlabel="Acción", ylabel="Número de decisiones",
           title="Distribución de acciones durante el entrenamiento")
    ax.grid(axis="y", alpha=.2)
    fig.tight_layout()
    figures["distribucion_acciones"] = fig
    plt.show()
else:
    print("El registro no contiene contadores de acciones.")


## Interpretación final

Responde con datos: ¿aumentó el éxito?, ¿disminuyeron las colisiones?, ¿la política depende demasiado de una acción?, ¿qué episodio marca un cambio sostenido?, ¿el modelo conserva su desempeño al evaluar sin exploración?

## Exportar tablas y figuras

Una ejecución crea un directorio nuevo bajo `results_notebooks/`. Puedes elegir el destino con `CARRITO_NOTEBOOK_OUTPUT`. La tabla `contraste_entrenamiento.csv` conserva las medias y `uso_acciones.csv` registra el conteo por acción cuando está disponible.


In [ ]:
if OUTPUT_DIR:
    output_dir = ROOT / Path(OUTPUT_DIR).expanduser()
    try:
        output_dir.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        raise FileExistsError(
            f"El destino ya existe: {output_dir}. Elige un directorio nuevo "
            "en CARRITO_NOTEBOOK_OUTPUT."
        ) from None
else:
    output_root = ROOT / "results_notebooks"
    output_root.mkdir(parents=True, exist_ok=True)
    output_dir = Path(mkdtemp(prefix="entrenamiento-", dir=output_root))

comparison.to_csv(output_dir / "contraste_entrenamiento.csv", index_label="metrica", mode="x")
if not usage.empty:
    usage.to_csv(output_dir / "uso_acciones.csv", mode="x")

for name, figure in figures.items():
    with (output_dir / f"{name}.png").open("xb") as output:
        figure.savefig(output, format="png", dpi=150, bbox_inches="tight")

print("Archivos exportados a")
print(output_dir)
for path in sorted(output_dir.iterdir()):
    print(f"- {path.name}")
